In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [26]:
portfolio = pd.read_csv('../1data/portfolio.csv')
profile = pd.read_csv('../1data/profile.csv')
transcript = pd.read_csv('../1data/transcript.csv')

merged = pd.read_csv('../1data/all_merged.csv')
promotion = pd.read_csv('../1data/promotion_df.csv')
normal = pd.read_csv('../1data/normal_flow_df.csv')

---

In [27]:
promotion.head()

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,5.0,bogo,5,5,...,M,33,2017-04-21,72000.0,8.57,1,1,1,0,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,2.0,discount,10,2,...,M,33,2017-04-21,72000.0,14.11,1,1,1,0,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,2.0,discount,10,2,...,M,33,2017-04-21,72000.0,10.27,1,0,1,0,0
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,3.0,discount,7,3,...,O,40,2018-01-09,57000.0,11.93,1,1,1,1,1
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,5.0,bogo,5,5,...,O,40,2018-01-09,57000.0,22.05,1,1,1,1,1


In [28]:
# bogo, discount만 필터링
bd_df = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

In [30]:
channels = ['web', 'email', 'mobile', 'social']

# 채널 조합 전체
channel_funnel_list = []
for channel in channels:
    temp = bd_df[bd_df[channel] == 1].copy()
    channel_funnel_list.append({
        'channel': channel,
        'total_offers': len(temp),
        'viewed': temp['is_viewed'].sum(),
        'completed': temp['is_completed'].sum(),
        'view_rate(%)': round(temp['is_viewed'].sum() / len(temp) * 100, 2),
        #'view_drop_rate(%)': round((1 - temp['is_viewed'].sum() / len(temp)) * 100, 2),
        'completion_rate(%)': round(temp['is_completed'].sum() / temp['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - temp['is_completed'].sum() / temp['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': temp['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(temp['is_normal_flow'].sum() / len(temp) * 100, 2)
    })

channel_funnel = pd.DataFrame(channel_funnel_list)
display(channel_funnel)

# 채널 x offer_type
channel_type_list = []
for channel in channels:
    temp = bd_df[bd_df[channel] == 1].copy()
    for offer_type, group in temp.groupby('offer_type'):
        channel_type_list.append({
            'channel': channel,
            'offer_type': offer_type,
            'total_offers': len(group),
            'viewed': group['is_viewed'].sum(),
            'completed': group['is_completed'].sum(),
            'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
            #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
            'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
            #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
            'normal_flow_cnt': group['is_normal_flow'].sum(),
            'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
        })

channel_type_funnel = pd.DataFrame(channel_type_list)
display(channel_type_funnel)

# 채널 x offer_id
channel_id_list = []
for channel in channels:
    temp = bd_df[bd_df[channel] == 1].copy()
    for offer_id, group in temp.groupby('offer_id'):
        channel_id_list.append({
            'channel': channel,
            'offer_id': offer_id,
            'total_offers': len(group),
            'viewed': group['is_viewed'].sum(),
            'completed': group['is_completed'].sum(),
            'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
            #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
            'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
            #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
            'normal_flow_cnt': group['is_normal_flow'].sum(),
            'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
        })

channel_id_funnel = pd.DataFrame(channel_id_list)
display(channel_id_funnel)

,channel,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
0,web,53809,40496,29891,75.26,73.81,20913,38.87
1,email,61520,47259,33579,76.82,71.05,23519,38.23
2,mobile,53745,44548,30159,82.89,67.70,22183,41.27
3,social,38323,36187,21788,94.43,60.21,17925,46.77


,channel,offer_type,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
0,web,bogo,22956,18825,11981,82.00,63.64,8412,36.64
1,web,discount,30853,21671,17910,70.24,82.65,12501,40.52
2,email,bogo,30667,25588,15669,83.44,61.24,11018,35.93
3,email,discount,30853,21671,17910,70.24,82.65,12501,40.52
4,mobile,bogo,30667,25588,15669,83.44,61.24,11018,35.93
5,mobile,discount,23078,18960,14490,82.16,76.42,11165,48.38
6,social,bogo,22939,21387,11315,93.23,52.91,8894,38.77
7,social,discount,15384,14800,10473,96.20,70.76,9031,58.70


,channel,offer_id,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
0,web,bogo_10_10_5,7623,7326,3331,96.10,45.47,2759,36.19
1,web,bogo_5_5_5,7605,7298,4296,95.96,58.87,3529,46.40
2,web,bogo_5_5_7,7728,4201,4354,54.36,103.64,2124,27.48
3,web,discount_10_2_10,7684,7412,5317,96.46,71.74,4643,60.42
4,web,discount_10_2_7,7694,4160,4017,54.07,96.56,2134,27.74
5,web,discount_20_5_10,7775,2711,3420,34.87,126.15,1336,17.18
6,web,discount_7_3_7,7700,7388,5156,95.95,69.79,4388,56.99
7,email,bogo_10_10_5,7623,7326,3331,96.10,45.47,2759,36.19
8,email,bogo_10_10_7,7711,6763,3688,87.71,54.53,2606,33.80
9,email,bogo_5_5_5,7605,7298,4296,95.96,58.87,3529,46.40


In [31]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)
display(bd_df['channel_combo'].value_counts())

channel_combo
web+email+mobile+social    30612
web+email+mobile           15422
web+email                   7775
email+mobile+social         7711
Name: count, dtype: int64

In [32]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)

# 채널 조합 전체
combo_funnel_list = []
for combo, group in bd_df.groupby('channel_combo'):
    combo_funnel_list.append({
        'channel_combo': combo,
        'total_offers': len(group),
        'viewed': group['is_viewed'].sum(),
        'completed': group['is_completed'].sum(),
        'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
        #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
        'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': group['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
    })

combo_funnel = pd.DataFrame(combo_funnel_list).sort_values('total_offers', ascending=False)
display(combo_funnel)

# 채널 조합 x offer_type
combo_type_list = []
for (combo, offer_type), group in bd_df.groupby(['channel_combo', 'offer_type']):
    combo_type_list.append({
        'channel_combo': combo,
        'offer_type': offer_type,
        'total_offers': len(group),
        'viewed': group['is_viewed'].sum(),
        'completed': group['is_completed'].sum(),
        'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
        #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
        'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': group['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
    })

combo_type_funnel = pd.DataFrame(combo_type_list).sort_values('total_offers', ascending=False)
display(combo_type_funnel)

# 채널 조합 x offer_id
combo_id_list = []
for (combo, offer_id), group in bd_df.groupby(['channel_combo', 'offer_id']):
    combo_id_list.append({
        'channel_combo': combo,
        'offer_id': offer_id,
        'total_offers': len(group),
        'viewed': group['is_viewed'].sum(),
        'completed': group['is_completed'].sum(),
        'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
        #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
        'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': group['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
    })

combo_id_funnel = pd.DataFrame(combo_id_list).sort_values('total_offers', ascending=False)
display(combo_id_funnel)

,channel_combo,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
3,web+email+mobile+social,30612,29424,18100,96.12,61.51,15319,50.04
2,web+email+mobile,15422,8361,8371,54.21,100.12,4258,27.61
1,web+email,7775,2711,3420,34.87,126.15,1336,17.18
0,email+mobile+social,7711,6763,3688,87.71,54.53,2606,33.80


,channel_combo,offer_type,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
5,web+email+mobile+social,discount,15384,14800,10473,96.20,70.76,9031,58.70
4,web+email+mobile+social,bogo,15228,14624,7627,96.03,52.15,6288,41.29
1,web+email,discount,7775,2711,3420,34.87,126.15,1336,17.18
2,web+email+mobile,bogo,7728,4201,4354,54.36,103.64,2124,27.48
0,email+mobile+social,bogo,7711,6763,3688,87.71,54.53,2606,33.80
3,web+email+mobile,discount,7694,4160,4017,54.07,96.56,2134,27.74


,channel_combo,offer_id,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
1,web+email,discount_20_5_10,7775,2711,3420,34.87,126.15,1336,17.18
2,web+email+mobile,bogo_5_5_7,7728,4201,4354,54.36,103.64,2124,27.48
0,email+mobile+social,bogo_10_10_7,7711,6763,3688,87.71,54.53,2606,33.80
7,web+email+mobile+social,discount_7_3_7,7700,7388,5156,95.95,69.79,4388,56.99
3,web+email+mobile,discount_10_2_7,7694,4160,4017,54.07,96.56,2134,27.74
6,web+email+mobile+social,discount_10_2_10,7684,7412,5317,96.46,71.74,4643,60.42
4,web+email+mobile+social,bogo_10_10_5,7623,7326,3331,96.10,45.47,2759,36.19
5,web+email+mobile+social,bogo_5_5_5,7605,7298,4296,95.96,58.87,3529,46.40


(정상 흐름)